# Honest Evaluation & Model Hardening

Implements Tier 1 + Tier 2 of the plan. Each section is independent; run top-to-bottom first, then you can rerun any section.

1. **Speaker-leakage scan** — detect cross-batch speaker overlap (biggest invisible-inflation risk)
2. **Load batches + base-model registry** — shared setup for everything below
3. **2-rotation × repeated 5×5-fold CV** with bootstrap threshold CIs — honest evaluation
4. **Per-batch score histograms** — visualise where batch shift lives
5. **Ensemble-of-seeds** on whisper_wp_xgb — cheap variance reduction
6. **PCA diagnostic** on 1024-d Whisper — is the full dim needed?
7. **WavLM variant comparison** — pre vs ft, whole vs seg
8. **Isotonic calibration** + reliability diagram
9. **Final winner recipe** with rotation spread

Outputs saved under `checkpoints_honest_eval/`.

In [ ]:
from pathlib import Path
import json, itertools, warnings, re, copy
import numpy as np
import pandas as pd
from collections import defaultdict

import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, roc_auc_score, average_precision_score)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

warnings.filterwarnings('ignore')

NB_DIR   = Path('.').resolve()
SAVE_DIR = NB_DIR / 'checkpoints_honest_eval'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BATCHES = ['audios2', 'audios4', 'audios5']

# Training prior-adjustment kept consistent with fusion notebook
DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE

# Evaluation knobs
N_SEEDS = 3     # repeated K-fold seeds (3 keeps runtime manageable; bump to 5 for final report)
N_FOLDS = 5
BOOT_N  = 200   # bootstrap iterations for threshold CI

STRATEGIES = ['F1','P80','P85','P90','P95']
PREC_FLOOR = {'P80':0.80, 'P85':0.85, 'P90':0.90, 'P95':0.95}

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

# 55 text features, same as text_cheating_detection.ipynb ALL_FEATURES
ALL_TEXT_FEATURES = [
    'filler_rate','filler_count','repetition_rate','repair_rate','discourse_marker_rate','hedge_rate',
    'ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
    'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
    'noun_rate','verb_rate','adj_rate',
    'pause_mean','pause_std','pause_median','pause_skew','long_pause_rate','pause_ratio','n_pauses',
    'pause_regularity','pause_before_content_ratio','pause_before_function_ratio','mid_phrase_pause_rate',
    'words_per_sec','articulation_rate','initial_pause','longest_pause',
    'suspicious_gap_count','suspicious_gap_ratio',
    'formal_transition_count','formal_transition_rate','ai_phrase_count','ai_phrase_rate',
    'f0_mean','f0_std','f0_range','f0_skew','f0_slope','energy_mean','energy_std','speaking_rate_std',
    'jitter_local','shimmer_local','hnr_mean',
    'mean_perplexity','burstiness',
]
STYLO_FEATS = ['ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
               'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
               'noun_rate','verb_rate','adj_rate']
TEXT_4GROUP = ALL_TEXT_FEATURES[:40]  # disfluency(6)+stylo(15)+pause(15)+formal_ai(4)? Defined precisely below

TEXT_4GROUP = [
    'filler_rate','filler_count','repetition_rate','repair_rate','discourse_marker_rate','hedge_rate',
    'ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
    'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
    'noun_rate','verb_rate','adj_rate',
    'pause_mean','pause_std','pause_median','pause_skew','long_pause_rate','pause_ratio','n_pauses',
    'pause_regularity','pause_before_content_ratio','pause_before_function_ratio','mid_phrase_pause_rate',
    'words_per_sec','articulation_rate','initial_pause','longest_pause',
    'formal_transition_count','formal_transition_rate','ai_phrase_count','ai_phrase_rate',
]

print(f'N_SEEDS={N_SEEDS}  N_FOLDS={N_FOLDS}  BOOT_N={BOOT_N}')
print(f'SPW_DEPLOY={SPW_DEPLOY:.2f}  DEPLOY_POS_RATE={DEPLOY_POS_RATE}')
print(f'matplotlib available: {HAS_MPL}')
print(f'Save dir: {SAVE_DIR}')

## 1. Speaker-leakage scan

If the same person appears in both CV and test (even across batches), we're measuring speaker memorisation, not cheating detection. This cell:

1. Looks at raw filenames across `audios2/`, `audios4/`, `audios5/` and prints a sample
2. Tries a few common patterns (name prefix, digit ID, `Q\d+` question code) to extract a speaker-like token
3. Reports overlap across batches

**If the auto-pattern doesn't match your filename convention**, tell me the naming scheme and I'll fix the regex. A no-op here (zero speakers extracted) is a *red flag*, not "all good".

In [ ]:
# Speaker-leakage scan v3 — uses GT filenames + the known structure
# Filenames are: <candidate_id>_<question_number>.<ext>
# (audio physically lives at audiosX/<candidate_id>/<candidate_id>_<Q>.wav)

def gt_filenames(batch):
    p = NB_DIR / f'{batch}GT.csv'
    if not p.exists(): return []
    gt = pd.read_csv(p)
    fn_col = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    return gt[fn_col].astype(str).tolist()

# Greedy match up to the last "_<digits>.<ext>" — that <digits> is the question number,
# everything before it is the candidate_id (works for "42_25.wav", "john_smith_25.wav", etc.)
CANDIDATE_RE = re.compile(r'^(.+)_(\d{1,3})\.[a-zA-Z0-9]+$')

def parse_candidate(fn):
    m = CANDIDATE_RE.match(fn)
    if not m: return None, None
    return m.group(1), m.group(2)

records = []
for b in BATCHES:
    files = gt_filenames(b)
    for fn in files:
        cid, q = parse_candidate(fn)
        records.append({'batch': b, 'filename': fn, 'candidate_id': cid, 'question': q})
rec_df = pd.DataFrame(records)

print('=== Speaker-leakage scan ===\n')
hit = rec_df['candidate_id'].notna()
print(f'Filenames parsed:  {hit.sum()} / {len(rec_df)}  ({hit.mean()*100:.1f}%)')
print('Sample parses (10):')
for _, r in rec_df.head(10).iterrows():
    print(f'   {r["batch"]}/{r["filename"]:35s}  -> candidate={r["candidate_id"]!r:25s}  q={r["question"]}')

print('\n--- Per-batch candidate counts ---')
print(f'{"batch":<10} {"audios":>7} {"candidates":>12} {"audios/cand (mean)":>20} {"questions":>15}')
for b in BATCHES:
    sub = rec_df[(rec_df['batch']==b) & rec_df['candidate_id'].notna()]
    n_audio = len(sub)
    n_cand  = sub['candidate_id'].nunique()
    avg_per = n_audio / max(n_cand, 1)
    qs = sorted(sub['question'].dropna().unique().tolist())
    print(f'{b:<10} {n_audio:>7} {n_cand:>12} {avg_per:>20.2f}  {qs}')

# --- Cross-batch overlap on candidate_id ---
print('\n--- Cross-batch candidate overlap (THE leakage check) ---')
valid = rec_df.dropna(subset=['candidate_id'])
cand_to_batches = valid.groupby('candidate_id')['batch'].apply(lambda s: sorted(set(s)))
shared = cand_to_batches[cand_to_batches.apply(len) > 1]
print(f'Candidates appearing in >1 batch: {len(shared)} / {valid["candidate_id"].nunique()} unique candidates')
if len(shared):
    print('  first 20 examples:')
    for cid, bs in shared.head(20).items():
        counts = {b: int(((valid["candidate_id"]==cid) & (valid["batch"]==b)).sum()) for b in bs}
        print(f'    {cid!r:30s}  in {bs}  counts={counts}')
    print('\n!! CROSS-BATCH LEAKAGE: same candidate in multiple batches.')
    print('   GroupKFold (next change) prevents within-batch leakage, but cross-batch')
    print('   leakage is in the train/test split itself — these candidates need to be')
    print('   removed from one side or assigned to one batch only.')
else:
    print('   No cross-batch candidate overlap. CV/test split is speaker-clean across batches.')

# --- Within-batch: every candidate should have exactly 3 audios (Q25/Q26/Q27) ---
print('\n--- Within-batch: candidates with unexpected audio counts ---')
for b in BATCHES:
    sub = valid[valid['batch']==b]
    counts = sub.groupby('candidate_id').size()
    weird = counts[counts != 3]
    if len(weird):
        print(f'  {b}: {len(weird)} candidates with != 3 audios')
        for cid, n in weird.head(5).items():
            print(f'    {cid}: {n} audios')
    else:
        print(f'  {b}: all candidates have exactly 3 audios')

# --- Attach candidate_id to the loaded batches DataFrames so Section 3 can use it ---
for b, df in batches.items():
    df['candidate_id'] = df['filename'].map(lambda f: parse_candidate(f)[0])
    df['question']     = df['filename'].map(lambda f: parse_candidate(f)[1])
    n_missing = df['candidate_id'].isna().sum()
    if n_missing:
        print(f'  WARN {b}: {n_missing} rows in batches DataFrame have unparseable filenames')

rec_df.to_csv(SAVE_DIR / 'candidate_id_table.csv', index=False)
print(f'\nSaved -> {SAVE_DIR / "candidate_id_table.csv"}')

## 2. Load all 3 batches + shared helpers

Reads the existing feature CSVs and returns one DataFrame per batch. Each DataFrame carries:
- `filename`, `label_int`, `batch`
- WavLM pretrained-whole columns (`wavlm_*`)
- Whisper columns (`whisper_*`)
- All 55 text features

Also defines the base-model registry and all evaluation helpers used in every section below.

In [ ]:
# ---------- Data loading ----------
def load_gt(name):
    gt = pd.read_csv(NB_DIR / f'{name}GT.csv')
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def _first_existing(candidates):
    """Return the first path that exists, or None."""
    for c in candidates:
        p = NB_DIR / c
        if p.exists(): return p
    return None

# Actual filename conventions in this project (mixed across notebooks):
#   whole pretrained: {batch}_wavlm_whole.csv     (also: {batch}_whole_pretrained.csv)
#   whole finetuned : {batch}_whole_finetuned.csv
#   seg pretrained  : {batch}_wavlm_seg.csv       (also: {batch}_seg_pretrained.csv)
#   seg finetuned   : {batch}_seg_finetuned.csv
WAVLM_CSV_CANDIDATES = {
    'wpre': lambda n: [f'{n}_wavlm_whole.csv',     f'{n}_whole_pretrained.csv'],
    'wft' : lambda n: [f'{n}_whole_finetuned.csv'],
    'spre': lambda n: [f'{n}_wavlm_seg.csv',       f'{n}_seg_pretrained.csv'],
    'sft' : lambda n: [f'{n}_seg_finetuned.csv'],
}

def load_folder(name):
    """Load every available feature CSV for a batch and merge into one DataFrame.
    Adds suffixes so wavlm pre/ft/seg cols don't clash with each other or whisper."""
    gt   = load_gt(name)
    text = pd.read_csv(NB_DIR / f'{name}_features.csv')
    df = gt.merge(text, on='filename', how='inner')

    found = {}
    for tag, cand_fn in WAVLM_CSV_CANDIDATES.items():
        path = _first_existing(cand_fn(name))
        if path is not None:
            found[tag] = path
            sub = pd.read_csv(path)
            # Suffix every non-filename column with _<tag> so no two wavlm CSVs collide
            sub = sub.rename(columns={c: (c if c == 'filename' else f'{c}_{tag}') for c in sub.columns})
            df = df.merge(sub, on='filename', how='inner')

    # Whisper whole (1024d)
    whr = pd.read_csv(NB_DIR / f'{name}_whisper_whole.csv')
    df = df.merge(whr.rename(columns={c: (c if c == 'filename' else f'{c}_whisper') for c in whr.columns}),
                  on='filename', how='inner')

    df['batch'] = name
    df.attrs['wavlm_paths'] = {k: str(v) for k, v in found.items()}
    return df

batches = {b: load_folder(b) for b in BATCHES}

print('=== Detected WavLM CSVs per batch ===')
for b in BATCHES:
    paths = batches[b].attrs.get('wavlm_paths', {})
    print(f'  {b}:')
    for tag in ['wpre','wft','spre','sft']:
        p = paths.get(tag)
        print(f'    {tag:5s} -> {p.split(chr(92))[-1] if p else "  (missing)"}')

# Discover column lists from the first batch
first = batches[BATCHES[0]]
WPRE_COLS = [c for c in first.columns if (c.startswith('wavlm_') or c.startswith('wavlm_mean_') or c.startswith('wavlm_std_'))
             and c.endswith('_wpre')]
WFT_COLS  = [c for c in first.columns if (c.startswith('wavlm_') or c.startswith('wavlm_mean_') or c.startswith('wavlm_std_'))
             and c.endswith('_wft')]
SPRE_COLS = [c for c in first.columns if (c.startswith('wavlm_mean_') or c.startswith('wavlm_std_'))
             and c.endswith('_spre')]
SFT_COLS  = [c for c in first.columns if (c.startswith('wavlm_mean_') or c.startswith('wavlm_std_'))
             and c.endswith('_sft')]
WH_COLS   = [c for c in first.columns if c.startswith('whisper_') and c.endswith('_whisper')]
TEXT_ALL   = [c for c in ALL_TEXT_FEATURES if c in first.columns]
TEXT_STYLO = [c for c in STYLO_FEATS       if c in first.columns]

print('\n=== Per-batch row counts ===')
for b, df in batches.items():
    y = df['label_int'].values
    print(f'  {b}: n={len(df):4d}  cheat={int((y==1).sum()):3d}  honest={int((y==0).sum()):3d}')
print(f'\n=== Feature dimensions ===')
print(f'  WavLM whole-pretrained (_wpre):  {len(WPRE_COLS)}')
print(f'  WavLM whole-finetuned  (_wft) :  {len(WFT_COLS)}')
print(f'  WavLM seg-pretrained   (_spre):  {len(SPRE_COLS)}')
print(f'  WavLM seg-finetuned    (_sft) :  {len(SFT_COLS)}')
print(f'  Whisper                       :  {len(WH_COLS)}')
print(f'  Text all / stylo              :  {len(TEXT_ALL)} / {len(TEXT_STYLO)}')

# ---------- XGBoost factories ----------
def make_xgb(n_feats, seed=42):
    colsample = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=seed)

def fit_score(X_tr, y_tr, X_te, seed=42):
    sc = StandardScaler().fit(X_tr)
    m  = make_xgb(X_tr.shape[1], seed=seed)
    m.fit(sc.transform(X_tr), y_tr)
    return m.predict_proba(sc.transform(X_te))[:,1]

# ---------- Threshold pickers ----------
def best_f1_thr(proba, y, grid=np.arange(0.20, 0.81, 0.01)):
    bt, bf = 0.5, -1.0
    for thr in grid:
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > bf: bf, bt = f, float(thr)
    return bt, bf

def best_rec_at_prec(proba, y, target, min_tp=3):
    best = None
    for thr in np.arange(0.99, 0.10, -0.01):
        pred = (proba >= thr).astype(int)
        cm = confusion_matrix(y, pred, labels=[0,1])
        if cm[1,1] < min_tp: continue
        p = precision_score(y, pred, zero_division=0)
        r = recall_score(y, pred, zero_division=0)
        if p >= target and (best is None or r > best[1]):
            best = (float(thr), float(r), float(p))
    return best if best is not None else (None, None, None)

def pick_thr(proba, y, strategy):
    if strategy == 'F1':
        thr, v = best_f1_thr(proba, y)
        return thr, v
    thr, rec, _ = best_rec_at_prec(proba, y, PREC_FLOOR[strategy])
    return thr, rec

def metrics_at(proba, y, thr):
    if thr is None: return dict(prec=None, rec=None, f1=None)
    pred = (proba >= thr).astype(int)
    return dict(
        prec=float(precision_score(y, pred, zero_division=0)),
        rec =float(recall_score(y, pred, zero_division=0)),
        f1  =float(f1_score(y, pred, zero_division=0)),
    )

# ---------- Base-model registry ----------
def mk_X(cols): return lambda d: d[cols].fillna(0).values

BASE_REGISTRY = {
    'whisper_wp_xgb': (mk_X(WH_COLS), lambda s=42: make_xgb(len(WH_COLS), s)),
}
if WPRE_COLS: BASE_REGISTRY['wavlm_whole_pre'] = (mk_X(WPRE_COLS), lambda s=42: make_xgb(len(WPRE_COLS), s))
if WFT_COLS:  BASE_REGISTRY['wavlm_whole_ft']  = (mk_X(WFT_COLS),  lambda s=42: make_xgb(len(WFT_COLS), s))
if SPRE_COLS: BASE_REGISTRY['wavlm_seg_pre']   = (mk_X(SPRE_COLS), lambda s=42: make_xgb(len(SPRE_COLS), s))
if SFT_COLS:  BASE_REGISTRY['wavlm_seg_ft']    = (mk_X(SFT_COLS),  lambda s=42: make_xgb(len(SFT_COLS), s))
BASE_REGISTRY['text_all']   = (mk_X(TEXT_ALL),   lambda s=42: make_xgb(len(TEXT_ALL), s))
BASE_REGISTRY['text_stylo'] = (mk_X(TEXT_STYLO), lambda s=42: make_xgb(len(TEXT_STYLO), s))

print(f'\nBase models in registry: {list(BASE_REGISTRY)}')

## 3. Repeated 5×5-fold CV × 2 rotations, with bootstrap threshold CI

Runs each base model under:
- **Rotation A**: train=[audios2+audios4], CV=audios4, test=audios5
- **Rotation B**: train=[audios2+audios5], CV=audios5, test=audios4

For each rotation, runs `N_SEEDS` × `N_FOLDS` = 15 OOF predictions (at default knobs). All 15 × N_CV predictions per sample are averaged into a single OOF vector per (seed, model) then *concatenated* across seeds for threshold selection.

For each strategy (F1, P80/85/90/95):
- Pick threshold on concatenated OOF
- Bootstrap-resample the (OOF, y) pair `BOOT_N` times → recompute threshold each time → report median + 10/90th percentiles
- Freeze the median threshold, apply to test → report test precision/recall + gap vs CV metric

Output: one table per rotation + a combined "rotation spread" table showing mean and std of test metrics across rotations.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

USE_GROUP_CV = True   # set False to fall back to plain StratifiedKFold (the leaky version)

def repeated_kfold_oof(df_cv, df_always, X_fn, factory, n_seeds=N_SEEDS, n_folds=N_FOLDS):
    """OOF averaged over n_seeds repeats of n_folds CV.
    If USE_GROUP_CV=True and df_cv has a 'candidate_id' column, splits by candidate
    so a candidate's Q25/Q26/Q27 never end up on both sides of a fold."""
    y_cv = df_cv['label_int'].values
    groups = df_cv['candidate_id'].values if (USE_GROUP_CV and 'candidate_id' in df_cv.columns) else None
    has_groups = groups is not None and pd.notna(groups).all()

    oof_sum = np.zeros(len(df_cv)); oof_cnt = np.zeros(len(df_cv))
    for s in range(n_seeds):
        if has_groups:
            splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=42 + s)
            split_iter = splitter.split(df_cv, y_cv, groups=groups)
        else:
            splitter = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42 + s)
            split_iter = splitter.split(df_cv, y_cv)
        for tr_idx, va_idx in split_iter:
            df_tr_fold = df_cv.iloc[tr_idx]
            df_va_fold = df_cv.iloc[va_idx]
            df_tr = pd.concat([df_always, df_tr_fold], ignore_index=True) if df_always is not None and len(df_always) else df_tr_fold
            Xtr = X_fn(df_tr); ytr = df_tr['label_int'].values
            Xva = X_fn(df_va_fold)
            sc = StandardScaler().fit(Xtr)
            clf = factory(42 + s)
            clf.fit(sc.transform(Xtr), ytr)
            p = clf.predict_proba(sc.transform(Xva))[:,1]
            oof_sum[va_idx] += p
            oof_cnt[va_idx] += 1
    return oof_sum / np.maximum(oof_cnt, 1)

def bootstrap_thr_ci(proba, y, strategy, n_boot=BOOT_N, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y); idxs = np.arange(n)
    thrs = []
    for _ in range(n_boot):
        b = rng.choice(idxs, size=n, replace=True)
        pb, yb = proba[b], y[b]
        if len(set(yb.tolist())) < 2: continue
        thr, _ = pick_thr(pb, yb, strategy)
        if thr is not None: thrs.append(thr)
    if not thrs:
        return None, None, None, 0
    t = np.array(thrs)
    return float(np.median(t)), float(np.percentile(t,10)), float(np.percentile(t,90)), len(t)

def run_rotation_base(train_folders, cv_target, test_folder, models=None):
    models = models or list(BASE_REGISTRY)
    df_cv  = batches[cv_target].reset_index(drop=True)
    df_al  = pd.concat([batches[b] for b in train_folders if b != cv_target], ignore_index=True) \
             if any(b != cv_target for b in train_folders) else None
    df_te  = batches[test_folder].reset_index(drop=True)
    y_cv   = df_cv['label_int'].values
    y_te   = df_te['label_int'].values

    rows = []
    oof_store = {}
    for name in models:
        X_fn, factory = BASE_REGISTRY[name]
        oof = repeated_kfold_oof(df_cv, df_al, X_fn, factory)
        oof_store[name] = oof

        # Refit on full train for test
        df_tr_full = pd.concat([df_al, df_cv], ignore_index=True) if df_al is not None and len(df_al) else df_cv
        Xtr = X_fn(df_tr_full); ytr = df_tr_full['label_int'].values
        Xte = X_fn(df_te)
        sc = StandardScaler().fit(Xtr)
        clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
        p_te = clf.predict_proba(sc.transform(Xte))[:,1]

        for s in STRATEGIES:
            thr_pt, cv_val = pick_thr(oof, y_cv, s)
            thr_md, thr_lo, thr_hi, _ = bootstrap_thr_ci(oof, y_cv, s)
            thr_use = thr_md if thr_md is not None else thr_pt
            te = metrics_at(p_te, y_te, thr_use)
            te_val = te['f1'] if s == 'F1' else te['rec']
            gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
            rows.append({
                'model': name, 'strategy': s,
                'thr_median': round(thr_use,3) if thr_use is not None else None,
                'thr_p10':    round(thr_lo,3)  if thr_lo  is not None else None,
                'thr_p90':    round(thr_hi,3)  if thr_hi  is not None else None,
                'cv':         round(cv_val,3)  if cv_val  is not None else None,
                'te':         round(te_val,3)  if te_val  is not None else None,
                'te_prec':    round(te['prec'],3) if te['prec'] is not None else None,
                'te_rec':     round(te['rec'],3)  if te['rec']  is not None else None,
                'gap':        round(gap,3)    if gap    is not None else None,
            })
    return pd.DataFrame(rows), oof_store

print(f'CV protocol: {"StratifiedGroupKFold (candidate-aware)" if USE_GROUP_CV else "StratifiedKFold (leaky)"}\n')
print('Running Rotation A: train=[a2,a4] CV=a4 test=a5 ...')
df_A, oof_A = run_rotation_base(['audios2','audios4'], 'audios4', 'audios5')
print('Running Rotation B: train=[a2,a5] CV=a5 test=a4 ...')
df_B, oof_B = run_rotation_base(['audios2','audios5'], 'audios5', 'audios4')

print('\n=== ROTATION A (CV=audios4  test=audios5) ===')
with pd.option_context('display.max_columns', None, 'display.width', 180):
    print(df_A.to_string(index=False, na_rep='  --'))

print('\n=== ROTATION B (CV=audios5  test=audios4) ===')
with pd.option_context('display.max_columns', None, 'display.width', 180):
    print(df_B.to_string(index=False, na_rep='  --'))

df_A.to_csv(SAVE_DIR / 'rotation_A.csv', index=False)
df_B.to_csv(SAVE_DIR / 'rotation_B.csv', index=False)

In [ ]:
# --- Rotation spread: mean ± std of test metrics across A and B per (model, strategy) ---
merged = (df_A[['model','strategy','cv','te','te_prec','te_rec','gap']].rename(columns=lambda c: c+'_A' if c not in ('model','strategy') else c)
          .merge(df_B[['model','strategy','cv','te','te_prec','te_rec','gap']].rename(columns=lambda c: c+'_B' if c not in ('model','strategy') else c),
                 on=['model','strategy']))

for m in ['cv','te','te_prec','te_rec','gap']:
    a = merged[f'{m}_A']; b = merged[f'{m}_B']
    merged[f'{m}_mean'] = ((a.astype(float) + b.astype(float)) / 2).round(3)
    merged[f'{m}_spread'] = (b.astype(float) - a.astype(float)).abs().round(3)

view_cols = ['model','strategy',
             'cv_A','cv_B','cv_mean',
             'te_A','te_B','te_mean','te_spread',
             'te_prec_A','te_prec_B',
             'gap_A','gap_B','gap_mean']
print('=== ROTATION SPREAD (A vs B) ===')
print('  te_spread = |A - B|.  If > 0.05, the model is batch-variance-dominated;')
print('  any single-rotation number is a coin flip.\n')
with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(merged[view_cols].to_string(index=False, na_rep='  --'))

merged.to_csv(SAVE_DIR / 'rotation_spread.csv', index=False)
print(f'\nSaved -> {SAVE_DIR / "rotation_spread.csv"}')

## 4. Per-batch score histograms

For each base model, plot histograms of positive-class probability scores:
- Split by true label (positive vs negative)
- Split by batch (audios2 / audios4 / audios5)

Each model is **refit once on audios2+audios4**, then we score *all three batches* (including the held-out audios5) just to see the full distribution. This is a diagnostic plot, not used for model selection.

If a plot is needed without matplotlib, the cell falls back to printing numeric summaries (median, IQR, top-score negatives per batch).

In [ ]:
# Refit each base model on audios2+audios4, score ALL 3 batches
df_tr = pd.concat([batches['audios2'], batches['audios4']], ignore_index=True)
score_table = {}
for name, (X_fn, factory) in BASE_REGISTRY.items():
    Xtr = X_fn(df_tr); ytr = df_tr['label_int'].values
    sc  = StandardScaler().fit(Xtr)
    clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
    per_batch = {}
    for b in BATCHES:
        X = X_fn(batches[b])
        per_batch[b] = clf.predict_proba(sc.transform(X))[:,1]
    score_table[name] = per_batch

# Numeric summary per (model, batch, label)
print('=== Score distribution summary ===\n')
print(f'{"model":<18} {"batch":<10} {"label":<6} {"n":>4} {"median":>8} {"p25":>7} {"p75":>7} {"p95neg":>9}')
for name in score_table:
    for b in BATCHES:
        y = batches[b]['label_int'].values
        p = score_table[name][b]
        for lbl, tag in [(0, 'neg'), (1, 'pos')]:
            mask = (y == lbl)
            if mask.sum() == 0: continue
            vals = p[mask]
            p95_neg = float(np.percentile(p[y==0], 95)) if (y==0).sum() > 0 else np.nan
            print(f'{name:<18} {b:<10} {tag:<6} {int(mask.sum()):>4} '
                  f'{np.median(vals):>8.3f} {np.percentile(vals,25):>7.3f} {np.percentile(vals,75):>7.3f} '
                  f'{p95_neg:>9.3f}')
    print()

# Matplotlib grid (fallback to text if unavailable)
if HAS_MPL:
    nmodels = len(score_table); nbatches = len(BATCHES)
    fig, axes = plt.subplots(nmodels, nbatches, figsize=(4*nbatches, 3*nmodels), sharex=True)
    if nmodels == 1: axes = axes[np.newaxis, :]
    for i, name in enumerate(score_table):
        for j, b in enumerate(BATCHES):
            ax = axes[i, j]
            y = batches[b]['label_int'].values
            p = score_table[name][b]
            if (y==0).sum() > 0: ax.hist(p[y==0], bins=25, alpha=0.5, label='honest', color='steelblue')
            if (y==1).sum() > 0: ax.hist(p[y==1], bins=25, alpha=0.5, label='cheat', color='orangered')
            ax.set_title(f'{name}  /  {b}', fontsize=9)
            ax.set_xlim(0, 1)
            if i == 0 and j == 0: ax.legend(fontsize=8)
    fig.suptitle('Score distributions per (model, batch).  Overlap in the right tail = hard negatives killing rec@P', fontsize=10)
    fig.tight_layout()
    fig.savefig(SAVE_DIR / 'score_histograms.png', dpi=110)
    plt.show()
    print(f'Saved histograms -> {SAVE_DIR / "score_histograms.png"}')
else:
    print('matplotlib unavailable; numeric summary above is the diagnostic.')

## 5. Ensemble-of-seeds on whisper_wp_xgb

XGBoost has stochastic column/row sampling. Averaging probabilities across 5 seeds reduces this variance at no extra data cost.

For Rotation A (train=[a2,a4], CV=a4, test=a5):
- Train whisper_wp_xgb with seeds 42..46 on the full train set, average test probabilities
- Compare F1 / gap to single-seed baseline (already in Section 3)

If the gap shrinks or te F1 rises, ensemble-of-seeds is a free win and should be the default.

In [ ]:
def ensemble_seeds_eval(train_folders, cv_target, test_folder, model_name, seeds=(42,43,44,45,46)):
    X_fn, factory = BASE_REGISTRY[model_name]
    df_cv = batches[cv_target].reset_index(drop=True)
    df_al = pd.concat([batches[b] for b in train_folders if b != cv_target], ignore_index=True)
    df_te = batches[test_folder].reset_index(drop=True)

    # CV OOF averaged across seeds (already produced by repeated_kfold_oof)
    oof = repeated_kfold_oof(df_cv, df_al, X_fn, factory, n_seeds=len(seeds), n_folds=N_FOLDS)
    y_cv = df_cv['label_int'].values
    y_te = df_te['label_int'].values

    # Single-seed test probabilities
    df_full = pd.concat([df_al, df_cv], ignore_index=True)
    Xtr = X_fn(df_full); ytr = df_full['label_int'].values
    Xte = X_fn(df_te)
    sc = StandardScaler().fit(Xtr)

    single = factory(42); single.fit(sc.transform(Xtr), ytr)
    p_te_single = single.predict_proba(sc.transform(Xte))[:,1]

    p_te_ensemble = np.zeros(len(df_te))
    for s in seeds:
        clf = factory(s); clf.fit(sc.transform(Xtr), ytr)
        p_te_ensemble += clf.predict_proba(sc.transform(Xte))[:,1]
    p_te_ensemble /= len(seeds)

    rows = []
    for tag, p_te in [('single seed=42', p_te_single), (f'ensemble of {len(seeds)} seeds', p_te_ensemble)]:
        for strat in STRATEGIES:
            thr, cv_val = pick_thr(oof, y_cv, strat)
            te = metrics_at(p_te, y_te, thr)
            te_val = te['f1'] if strat == 'F1' else te['rec']
            gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
            rows.append({
                'variant': tag, 'strategy': strat,
                'thr': round(thr,3) if thr is not None else None,
                'cv':  round(cv_val,3) if cv_val is not None else None,
                'te':  round(te_val,3) if te_val is not None else None,
                'te_prec': round(te['prec'],3) if te['prec'] is not None else None,
                'gap': round(gap,3) if gap is not None else None,
            })
    return pd.DataFrame(rows)

print('Rotation A  — ensemble-of-seeds on whisper_wp_xgb')
df_ens = ensemble_seeds_eval(['audios2','audios4'], 'audios4', 'audios5', 'whisper_wp_xgb')
with pd.option_context('display.max_columns', None, 'display.width', 160):
    print(df_ens.to_string(index=False, na_rep='  --'))
df_ens.to_csv(SAVE_DIR / 'ensemble_seeds.csv', index=False)

## 6. PCA diagnostic on Whisper 1024-d features

If Whisper features are noisy/redundant, PCA to fewer dims may *improve* generalisation (smaller gap) while keeping F1. If F1 drops at 80%-variance, information density is actually spread across many dims and full 1024d is needed.

Runs for Rotation A:
- **Baseline**: XGB on full 1024 Whisper dims (= whisper_wp_xgb from registry)
- **PCA@80% / 90% / 95%** variance: PCA fit on the *training* split inside each fold, transform CV and test, train XGB

Reports F1 / P85 / P90 / gap at each dim level. Comparison: does reduction help or hurt?

In [ ]:
def run_pca_rotation(train_folders, cv_target, test_folder, feat_cols, variances=(0.80, 0.90, 0.95)):
    df_cv = batches[cv_target].reset_index(drop=True)
    df_al = pd.concat([batches[b] for b in train_folders if b != cv_target], ignore_index=True)
    df_te = batches[test_folder].reset_index(drop=True)
    y_cv  = df_cv['label_int'].values
    y_te  = df_te['label_int'].values

    X_al = df_al[feat_cols].fillna(0).values
    X_cv = df_cv[feat_cols].fillna(0).values
    X_te = df_te[feat_cols].fillna(0).values
    y_al = df_al['label_int'].values

    rows = []
    for var_target in variances:
        # Repeated K-fold OOF with fold-scoped PCA
        oof_sum = np.zeros(len(X_cv)); oof_cnt = np.zeros(len(X_cv))
        for s in range(N_SEEDS):
            skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42 + s)
            for tr_i, va_i in skf.split(X_cv, y_cv):
                Xtr = np.vstack([X_al, X_cv[tr_i]]) if len(X_al) else X_cv[tr_i]
                ytr = np.concatenate([y_al, y_cv[tr_i]]) if len(X_al) else y_cv[tr_i]
                sc  = StandardScaler().fit(Xtr)
                pca = PCA(n_components=var_target, random_state=42).fit(sc.transform(Xtr))
                Ztr = pca.transform(sc.transform(Xtr))
                Zva = pca.transform(sc.transform(X_cv[va_i]))
                clf = make_xgb(Ztr.shape[1], seed=42 + s)
                clf.fit(Ztr, ytr)
                oof_sum[va_i] += clf.predict_proba(Zva)[:,1]
                oof_cnt[va_i] += 1
        oof = oof_sum / np.maximum(oof_cnt, 1)

        # Full-train refit → test
        Xtr = np.vstack([X_al, X_cv]) if len(X_al) else X_cv
        ytr = np.concatenate([y_al, y_cv]) if len(X_al) else y_cv
        sc  = StandardScaler().fit(Xtr)
        pca = PCA(n_components=var_target, random_state=42).fit(sc.transform(Xtr))
        Ztr = pca.transform(sc.transform(Xtr))
        Zte = pca.transform(sc.transform(X_te))
        clf = make_xgb(Ztr.shape[1], seed=42); clf.fit(Ztr, ytr)
        p_te = clf.predict_proba(Zte)[:,1]
        n_dims = pca.n_components_

        for strat in STRATEGIES:
            thr, cv_val = pick_thr(oof, y_cv, strat)
            te = metrics_at(p_te, y_te, thr)
            te_val = te['f1'] if strat == 'F1' else te['rec']
            gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
            rows.append({
                'variant': f'PCA@{int(var_target*100)}% ({n_dims}d)',
                'strategy': strat,
                'cv': round(cv_val,3) if cv_val is not None else None,
                'te': round(te_val,3) if te_val is not None else None,
                'te_prec': round(te['prec'],3) if te['prec'] is not None else None,
                'gap': round(gap,3) if gap is not None else None,
            })

    # Baseline (no PCA) from stored rotation A
    base = df_A[df_A['model']=='whisper_wp_xgb'][['strategy','cv','te','te_prec','gap']].copy()
    base['variant'] = f'no PCA ({len(feat_cols)}d)'
    rows_base = base[['variant','strategy','cv','te','te_prec','gap']].to_dict('records')
    return pd.DataFrame(rows_base + rows)

print('Rotation A — PCA sweep on Whisper features')
df_pca = run_pca_rotation(['audios2','audios4'], 'audios4', 'audios5', WH_COLS)
with pd.option_context('display.max_columns', None, 'display.width', 160):
    print(df_pca.to_string(index=False, na_rep='  --'))
df_pca.to_csv(SAVE_DIR / 'pca_diagnostic.csv', index=False)

## 7. WavLM variant comparison (pretrained/finetuned × whole/segmented)

`wavlm_wp` in our base registry is just the pretrained-whole variant. The 4-way notebook has 3 more:
- `whole_finetuned` (768d, fine-tuned WavLM, mean-pool)
- `seg_pretrained` (1536d, pretrained WavLM, segmented mean+std)
- `seg_finetuned` (1536d, fine-tuned, segmented mean+std)

This cell runs repeated-CV on all 4 for Rotation A and picks the best. If one of the fine-tuned/segmented variants has a smaller gap at P85/P90 than `whole_pretrained`, we should swap the default WavLM in the fusion recipe.

**Requires** the `{batch}_whole_pretrained.csv`, `{batch}_whole_finetuned.csv`, `{batch}_seg_pretrained.csv`, `{batch}_seg_finetuned.csv` files to exist (produced by `wavlm_4way_comparison.ipynb`).

In [ ]:
def load_wavlm_variant(batch, csv_suffix):
    p = NB_DIR / f'{batch}_{csv_suffix}.csv'
    if not p.exists():
        return None
    return pd.read_csv(p)

WAVLM_VARIANTS = {
    'whole_pretrained': 'wavlm_',          # 768d
    'whole_finetuned':  'wavlm_',          # 768d
    'seg_pretrained':   'wavlm_mean_|wavlm_std_',  # 1536d (mean+std)
    'seg_finetuned':    'wavlm_mean_|wavlm_std_',
}

def build_variant_df(batch, variant_key):
    feat = load_wavlm_variant(batch, variant_key)
    if feat is None: return None
    gt   = load_gt(batch)
    merged = feat.merge(gt, on='filename', how='inner')
    merged['batch'] = batch
    return merged

def run_wavlm_variant_rotation(variant_key, train_folders, cv_target, test_folder):
    dfs = {b: build_variant_df(b, variant_key) for b in set(train_folders + [test_folder])}
    if any(v is None for v in dfs.values()):
        return None, None
    feat_cols = [c for c in dfs[train_folders[0]].columns if c.startswith('wavlm_')]
    df_cv = dfs[cv_target].reset_index(drop=True)
    df_al = pd.concat([dfs[b] for b in train_folders if b != cv_target], ignore_index=True)
    df_te = dfs[test_folder].reset_index(drop=True)
    X_fn  = mk_X(feat_cols)
    factory = lambda s=42: make_xgb(len(feat_cols), s)

    oof = repeated_kfold_oof(df_cv, df_al, X_fn, factory)
    y_cv = df_cv['label_int'].values

    df_full = pd.concat([df_al, df_cv], ignore_index=True)
    Xtr = X_fn(df_full); ytr = df_full['label_int'].values
    Xte = X_fn(df_te);   y_te = df_te['label_int'].values
    sc = StandardScaler().fit(Xtr)
    clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
    p_te = clf.predict_proba(sc.transform(Xte))[:,1]

    rows = []
    for strat in STRATEGIES:
        thr, cv_val = pick_thr(oof, y_cv, strat)
        te = metrics_at(p_te, y_te, thr)
        te_val = te['f1'] if strat == 'F1' else te['rec']
        gap = (cv_val - te_val) if (cv_val is not None and te_val is not None) else None
        rows.append({
            'variant': f'{variant_key} ({len(feat_cols)}d)',
            'strategy': strat,
            'cv': round(cv_val,3) if cv_val is not None else None,
            'te': round(te_val,3) if te_val is not None else None,
            'te_prec': round(te['prec'],3) if te['prec'] is not None else None,
            'gap': round(gap,3) if gap is not None else None,
        })
    return pd.DataFrame(rows), oof

all_variant_rows = []
for vk in WAVLM_VARIANTS:
    print(f'Running variant: {vk} ...')
    dfv, _ = run_wavlm_variant_rotation(vk, ['audios2','audios4'], 'audios4', 'audios5')
    if dfv is None:
        print(f'  skipped ({vk} CSVs not found)')
        continue
    all_variant_rows.append(dfv)

if all_variant_rows:
    df_wvariants = pd.concat(all_variant_rows, ignore_index=True)
    print('\n=== WavLM variant comparison (Rotation A) ===')
    with pd.option_context('display.max_columns', None, 'display.width', 160):
        print(df_wvariants.to_string(index=False, na_rep='  --'))
    df_wvariants.to_csv(SAVE_DIR / 'wavlm_variants.csv', index=False)
else:
    print('No WavLM variant CSVs found. Run wavlm_4way_comparison.ipynb first to produce them.')

## 8. Isotonic calibration + reliability diagram

Fit isotonic on the repeated-CV OOF of whisper_wp_xgb (the strongest base), then apply to test.

**Check 1 — reliability diagram.** Bucket test predictions by calibrated score and compare to actual positive rate in each bucket. A good calibrator has actual ≈ calibrated within ±10 pts.

**Check 2 — recomputed thresholds on calibrated scores.** For each strategy, pick threshold on calibrated CV OOF, apply to calibrated test. Compare gap to the uncalibrated version from Section 3.

A meaningful gap reduction means calibration is worth shipping. A flat result means XGB is already well-calibrated and isotonic adds nothing.

In [ ]:
cal_target = 'whisper_wp_xgb'
oof = oof_A[cal_target]
y_cv_A = batches['audios4']['label_int'].values

# Refit single-seed on full train for test proba
X_fn, factory = BASE_REGISTRY[cal_target]
df_full = pd.concat([batches['audios2'], batches['audios4']], ignore_index=True)
Xtr = X_fn(df_full); ytr = df_full['label_int'].values
Xte = X_fn(batches['audios5']); y_te_A = batches['audios5']['label_int'].values
sc = StandardScaler().fit(Xtr)
clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
p_te = clf.predict_proba(sc.transform(Xte))[:,1]

iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y_cv_A)
p_cv_cal = iso.transform(oof)
p_te_cal = iso.transform(p_te)

# Reliability diagram — test side
bins = np.linspace(0, 1, 11)
bucket_ids = np.digitize(p_te_cal, bins) - 1
bucket_ids = np.clip(bucket_ids, 0, 9)
print('=== Reliability on test (audios5) — whisper_wp_xgb after isotonic ===')
print(f'{"bucket":<15} {"n":>4} {"mean_cal":>9} {"actual_pos_rate":>16}')
for b in range(10):
    m = bucket_ids == b
    if m.sum() == 0: continue
    print(f'[{bins[b]:.1f}-{bins[b+1]:.1f})    {int(m.sum()):>4} {float(p_te_cal[m].mean()):>9.3f} {float(y_te_A[m].mean()):>16.3f}')

# Threshold comparison (raw vs calibrated) per strategy
cmp_rows = []
for strat in STRATEGIES:
    # raw
    thr_r, cv_r = pick_thr(oof, y_cv_A, strat)
    te_r = metrics_at(p_te, y_te_A, thr_r)
    # calibrated
    thr_c, cv_c = pick_thr(p_cv_cal, y_cv_A, strat)
    te_c = metrics_at(p_te_cal, y_te_A, thr_c)
    # metric comparison: F1 or recall
    raw_te  = te_r['f1'] if strat == 'F1' else te_r['rec']
    cal_te  = te_c['f1'] if strat == 'F1' else te_c['rec']
    gap_raw = (cv_r - raw_te) if (cv_r is not None and raw_te is not None) else None
    gap_cal = (cv_c - cal_te) if (cv_c is not None and cal_te is not None) else None
    cmp_rows.append({
        'strategy': strat,
        'raw_thr': round(thr_r,3) if thr_r is not None else None,
        'raw_cv': round(cv_r,3) if cv_r is not None else None,
        'raw_te': round(raw_te,3) if raw_te is not None else None,
        'raw_te_prec': round(te_r['prec'],3) if te_r['prec'] is not None else None,
        'raw_gap': round(gap_raw,3) if gap_raw is not None else None,
        'cal_thr': round(thr_c,3) if thr_c is not None else None,
        'cal_cv': round(cv_c,3) if cv_c is not None else None,
        'cal_te': round(cal_te,3) if cal_te is not None else None,
        'cal_te_prec': round(te_c['prec'],3) if te_c['prec'] is not None else None,
        'cal_gap': round(gap_cal,3) if gap_cal is not None else None,
        'gap_delta': round(abs(gap_cal) - abs(gap_raw), 3) if (gap_raw is not None and gap_cal is not None) else None,
    })
cmp_df = pd.DataFrame(cmp_rows)
print('\n=== Raw vs calibrated thresholds on whisper_wp_xgb (Rotation A) ===')
print('  gap_delta < 0  = calibration reduced |gap|  (good).')
with pd.option_context('display.max_columns', None, 'display.width', 180):
    print(cmp_df.to_string(index=False, na_rep='  --'))
cmp_df.to_csv(SAVE_DIR / 'calibration_compare.csv', index=False)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(5,4))
    centers = (bins[:-1] + bins[1:]) / 2
    actual = []
    mean_cal = []
    for b in range(10):
        m = bucket_ids == b
        if m.sum() == 0:
            actual.append(np.nan); mean_cal.append(np.nan)
        else:
            actual.append(float(y_te_A[m].mean()))
            mean_cal.append(float(p_te_cal[m].mean()))
    ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='ideal')
    ax.plot(mean_cal, actual, 'o-', color='orangered', label='test')
    ax.set_xlabel('calibrated score (bucket mean)'); ax.set_ylabel('actual positive rate in bucket')
    ax.set_title(f'Reliability diagram — {cal_target} after isotonic (Rotation A)')
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(SAVE_DIR / 'reliability.png', dpi=110)
    plt.show()
    print(f'Saved -> {SAVE_DIR / "reliability.png"}')

## 9. Final winner recipe

Prints a compact decision summary:
1. Best *single* base model by average test F1 across both rotations
2. Best base model by P85 test recall (honest high-precision winner)
3. Flag any model where rotation-spread > 0.05 (batch-variance dominated)
4. Short verdict on ensemble, PCA, calibration, WavLM variant — did each help?

In [ ]:
print('='*90)
print('FINAL WINNER RECIPE (from Section 3 rotation spread)')
print('='*90)

# (1) + (2) — best single base by F1 and by P85 recall, averaged across rotations
f1_view = merged[merged['strategy']=='F1'][['model','te_mean','te_spread','gap_mean']]
best_f1 = f1_view.sort_values('te_mean', ascending=False).iloc[0]
print(f'\nBest single model by mean test F1: {best_f1["model"]}')
print(f'   te_F1 mean={best_f1["te_mean"]}  spread={best_f1["te_spread"]}  gap={best_f1["gap_mean"]}')

p85_view = merged[merged['strategy']=='P85'][['model','te_mean','te_spread','gap_mean','te_prec_A','te_prec_B']].copy()
# Only consider models where test precision actually held (>=0.80 on at least one rotation)
p85_view['held_precision'] = ((p85_view['te_prec_A'].astype(float) >= 0.80) |
                               (p85_view['te_prec_B'].astype(float) >= 0.80))
p85_candidates = p85_view[p85_view['held_precision']]
if len(p85_candidates):
    best_p85 = p85_candidates.sort_values('te_mean', ascending=False).iloc[0]
    print(f'\nBest single model by mean test rec@P85 (with precision holding): {best_p85["model"]}')
    print(f'   te_rec mean={best_p85["te_mean"]}  spread={best_p85["te_spread"]}  gap={best_p85["gap_mean"]}')
else:
    print('\nNo single base model held precision ≥ 0.80 on either rotation at P85 strategy.')

# (3) — batch-variance-dominated flag
print('\nModels with te_spread > 0.05 at ANY strategy (batch-variance dominated):')
unstable = merged[merged['te_spread'].astype(float) > 0.05][['model','strategy','te_A','te_B','te_spread']]
if len(unstable):
    with pd.option_context('display.max_columns', None, 'display.width', 140):
        print(unstable.to_string(index=False, na_rep='  --'))
else:
    print('   (none — all models transfer consistently across rotations)')

# (4) — short verdict on each experiment
print('\n' + '-'*90)
print('EXPERIMENT VERDICTS (compare numbers above)')
print('-'*90)
print('  Ensemble-of-seeds (Section 5): read df_ens above — compare single-seed vs 5-seed rows.')
print('     If 5-seed te F1 > single te F1 by >=0.01  -> adopt as default.')
print('  PCA diagnostic (Section 6): read df_pca — compare "no PCA" vs PCA@80/90/95%.')
print('     If PCA@80% matches no-PCA te F1 within 0.01 AND gap is smaller -> adopt PCA@80%.')
print('  WavLM variants (Section 7): if any variant beats whole_pretrained on gap @ P85 by >=0.05,')
print('     swap it in for fusion recipes.')
print('  Calibration (Section 8): if cmp_df shows gap_delta < 0 on >=3 strategies,')
print('     ship the isotonic calibrator alongside the model.')

print('\n' + '='*90)
print('NEXT STEP suggestions (once you send the numbers back)')
print('='*90)
print('  - If Tier-2 wins are small, move to Tier-3 (audio augmentation).')
print('  - If speaker-leakage scan flagged cross-batch tokens, redo all splits group-aware')
print('    and re-run this notebook — expect honest numbers to drop meaningfully.')
print('  - If whisper_wp_xgb is already at ~0.80 F1 with 0.05 gap, Tier-3 may only add a few points.')

print('\nAll tables saved to:', SAVE_DIR)

## 10. Data-asymmetry diagnostic — audios2-only training

Goal: discriminate between three hypotheses for why Rotation B (CV=audios5, test=audios4) shows a +0.13 to +0.19 F1 gap on whisper-driven fusions, while Rotation A (CV=audios4, test=audios5) shows ~0 gap.

The three hypotheses:
1. **a5 contaminates training** (label noise or batch-shortcut memorisation): whisper learns from a5 a rule that doesn't fit a4.
2. **a4 cheaters are intrinsically subtler**: whisper trained without ever seeing a4 still cannot find them.
3. **a5 is a narrower slice of cheating styles**: whisper trained on a2+a5 only sees one mode, but a4 has both modes.

Method: train whisper (and fine-tuned WavLM if available) on **audios2 alone** (147 positives / 72 negatives, ~67% positive — the inverted prior is real and noted), then score audios4 and audios5 separately. Use prior-invariant metrics (AUC, average precision, best-F1-over-threshold-sweep) so the audios2 67% training prior does not confound the comparison.

Reading the result:
- If `AUC(a4) ≈ AUC(a5)` and both are reasonable (>0.75), a5 is fine on its own — the Rot B gap comes from *training with a5*, pointing to label noise or batch-shortcut memorisation. → hypothesis 1.
- If `AUC(a4) << AUC(a5)`, a4 cheaters are harder to find from a2-only features. → hypothesis 2 or 3 (the balanced re-run distinguishes them from a prior issue).
- The score-distribution table below shows per-batch class separation. If `separation(a4) << separation(a5)`, whisper's features carry less signal on a4.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

if 'batches' not in dir() or 'BASE_REGISTRY' not in dir():
    raise RuntimeError("Run sections 1-3 first so 'batches' and 'BASE_REGISTRY' are in scope.")

# Sanity print the actual audios2 distribution (user flagged 147+/~70-).
_y2 = batches['audios2']['label_int'].values
print(f'audios2 training prior: {int((_y2==1).sum())}+ / {int((_y2==0).sum())}- '
      f'({(_y2==1).mean()*100:.1f}% positive)  — note: deployment prior is ~17% positive')

def _best_f1_sweep(p, y, grid=np.arange(0.05, 0.96, 0.01)):
    bf, bt = -1.0, 0.5
    for thr in grid:
        f = f1_score(y, (p >= thr).astype(int), zero_division=0)
        if f > bf: bf, bt = f, float(thr)
    return bt, bf

def _train_on_audios2(name, X_fn, factory, balanced=False, seed=42):
    df_tr = batches['audios2']
    if balanced:
        n_neg = int((df_tr['label_int']==0).sum())
        n_pos_target = n_neg  # match the smaller class so 1:1
        pos = df_tr[df_tr['label_int']==1].sample(n=n_pos_target, random_state=seed)
        neg = df_tr[df_tr['label_int']==0]
        df_tr = pd.concat([pos, neg], ignore_index=True)
    Xtr = X_fn(df_tr); ytr = df_tr['label_int'].values
    sc = StandardScaler().fit(Xtr)
    clf = factory(seed)
    clf.fit(sc.transform(Xtr), ytr)
    return clf, sc, ytr

def diagnose(name, X_fn, factory, balanced=False):
    clf, sc, ytr = _train_on_audios2(name, X_fn, factory, balanced=balanced)
    tag = 'BALANCED audios2' if balanced else 'audios2 (full, SPW_DEPLOY)'
    print(f'\n{"="*82}')
    print(f' {name}   trained on {tag}  ({int((ytr==1).sum())}+/{int((ytr==0).sum())}-)')
    print('='*82)
    rows = []
    for tgt in ['audios2','audios4','audios5']:
        df_e = batches[tgt]
        Xe   = X_fn(df_e); ye = df_e['label_int'].values
        p    = clf.predict_proba(sc.transform(Xe))[:,1]
        auc  = roc_auc_score(ye, p) if len(set(ye.tolist())) > 1 else float('nan')
        ap   = average_precision_score(ye, p) if len(set(ye.tolist())) > 1 else float('nan')
        thr, f1 = _best_f1_sweep(p, ye)
        rows.append({
            'eval_batch':    tgt,
            'n':             len(ye),
            'pos':           int((ye==1).sum()),
            'pos_rate':      round((ye==1).mean(), 3),
            'AUC':           round(auc, 3),
            'AP':            round(ap, 3),
            'best_thr':      round(thr, 2),
            'best_F1':       round(f1, 3),
            'pos_score_med': round(np.median(p[ye==1]), 3) if (ye==1).sum() else None,
            'neg_score_med': round(np.median(p[ye==0]), 3) if (ye==0).sum() else None,
        })
    df = pd.DataFrame(rows)
    df['separation'] = (df['pos_score_med'] - df['neg_score_med']).round(3)
    with pd.option_context('display.max_columns', None, 'display.width', 160):
        print(df.to_string(index=False))
    return df

# ---- Run for whisper (always present) and fine-tuned WavLM (if loaded) ----
print('\n' + '#'*82)
print('# PASS 1: train on FULL audios2 (147+/72-, the 67% positive training prior)')
print('# Uses SPW_DEPLOY=4.88 to calibrate to deployment 17% prior.')
print('#'*82)
res_w_full = diagnose('whisper_wp_xgb',
                       BASE_REGISTRY['whisper_wp_xgb'][0],
                       BASE_REGISTRY['whisper_wp_xgb'][1],
                       balanced=False)
if 'wavlm_whole_ft' in BASE_REGISTRY:
    res_v_full = diagnose('wavlm_whole_ft',
                          BASE_REGISTRY['wavlm_whole_ft'][0],
                          BASE_REGISTRY['wavlm_whole_ft'][1],
                          balanced=False)

print('\n' + '#'*82)
print('# PASS 2: BALANCED audios2 (down-sample positives to 72 to remove the 67% prior)')
print('# Removes any prior-shift confound from the audios2-only training.')
print('#'*82)
res_w_bal  = diagnose('whisper_wp_xgb',
                       BASE_REGISTRY['whisper_wp_xgb'][0],
                       BASE_REGISTRY['whisper_wp_xgb'][1],
                       balanced=True)
if 'wavlm_whole_ft' in BASE_REGISTRY:
    res_v_bal = diagnose('wavlm_whole_ft',
                         BASE_REGISTRY['wavlm_whole_ft'][0],
                         BASE_REGISTRY['wavlm_whole_ft'][1],
                         balanced=True)

# ---- Compact summary that maps directly onto the three hypotheses ----
print('\n' + '='*82)
print(' SUMMARY — does whisper-trained-on-a2-alone find a4 cheaters as well as a5?')
print('='*82)
def _row(label, df):
    a4 = df[df['eval_batch']=='audios4'].iloc[0]
    a5 = df[df['eval_batch']=='audios5'].iloc[0]
    return {
        'pass':            label,
        'AUC_a4':          a4['AUC'],
        'AUC_a5':          a5['AUC'],
        'dAUC(a4-a5)':     round(a4['AUC'] - a5['AUC'], 3),
        'AP_a4':           a4['AP'],
        'AP_a5':           a5['AP'],
        'sep_a4':          a4['separation'],
        'sep_a5':          a5['separation'],
        'bestF1_a4':       a4['best_F1'],
        'bestF1_a5':       a5['best_F1'],
    }
summary_rows = [_row('whisper · full a2',     res_w_full),
                _row('whisper · balanced a2', res_w_bal)]
if 'wavlm_whole_ft' in BASE_REGISTRY:
    summary_rows += [_row('wavlm_ft · full a2',     res_v_full),
                     _row('wavlm_ft · balanced a2', res_v_bal)]
summary_df = pd.DataFrame(summary_rows)
with pd.option_context('display.max_columns', None, 'display.width', 160):
    print(summary_df.to_string(index=False))

print('\nDecision rules:')
print('  (1) dAUC(a4-a5) ≈ 0  AND both AUCs > 0.75   ->  a5 is fine on its own.')
print('      The Rot B gap comes from TRAINING with a5 -> label noise or batch')
print('      memorisation.  Action: spot-check a5 positives for label correctness.')
print('  (2) dAUC(a4-a5) << 0  (a4 much harder)      ->  a4 cheating is subtler.')
print('      Action: a4 must always be in training; never CV with a5 alone.')
print('  (3) AUC similar but BOTH low (< 0.65)       ->  audios2 features do not')
print('      generalise off-batch at all -> per-batch standardisation or domain')
print('      adaptation is the next move, not relabelling.')


## 11. audios5 label-suspicion audit

§10 confirmed: training on audios2 alone gives AUC 0.92 on a4 and 0.94 on a5. Adding a5 to training is what breaks a4 transfer. This cell ranks individual a5 audios by how strongly an a2-only-trained model disagrees with their label.

The top of each disagreement list (positives the model thinks are honest, negatives the model thinks are cheating) is the shortlist worth a 30-second re-listen. Two acoustic models trained independently on a2 (whisper and fine-tuned WavLM) — agreement between their "this label looks wrong" signal is much stronger evidence than either alone.

Output: per-class shortlists (top-15 each) and a combined "both models flag this" list. Filenames are printed so you can pull the audio file directly.

In [ ]:
if 'batches' not in dir() or 'BASE_REGISTRY' not in dir():
    raise RuntimeError('Run sections 1-3 first.')
if '_train_on_audios2' not in dir():
    raise RuntimeError('Run section 10 first (defines _train_on_audios2).')

import re as _re_audit
_RE_CAND = _re_audit.compile(r'^(.+)_(\d{1,3})\.[a-zA-Z0-9]+$')
for _b in ['audios2','audios4','audios5']:
    if 'candidate_id' not in batches[_b].columns:
        batches[_b]['candidate_id'] = batches[_b]['filename'].astype(str).map(
            lambda f: (_RE_CAND.match(f).group(1) if _RE_CAND.match(f) else None))

# ---------- Score audios5 with both acoustic models trained on a2 only ----------
def _score_audios5(name):
    X_fn, factory = BASE_REGISTRY[name]
    clf, sc, _ = _train_on_audios2(name, X_fn, factory, balanced=False)
    df = batches['audios5'].copy()
    p = clf.predict_proba(sc.transform(X_fn(df)))[:, 1]
    return df[['filename','candidate_id','label_int']].assign(proba=p)

w = _score_audios5('whisper_wp_xgb')
v = _score_audios5('wavlm_whole_ft') if 'wavlm_whole_ft' in BASE_REGISTRY else None
if v is None:
    raise RuntimeError('wavlm_whole_ft not in BASE_REGISTRY — joint audit needs both models.')

# ---------- Build joint table with audio_path ----------
TARGET_BATCH = 'audios5'
def _audio_path(cid, fn):
    return str(NB_DIR / TARGET_BATCH / str(cid) / str(fn))

joint = w.rename(columns={'proba':'whisper'}).merge(
    v[['filename','proba']].rename(columns={'proba':'wavlm_ft'}),
    on='filename')
joint['avg']      = (joint['whisper'] + joint['wavlm_ft']) / 2
joint['min_prob'] = joint[['whisper','wavlm_ft']].min(axis=1)
joint['max_prob'] = joint[['whisper','wavlm_ft']].max(axis=1)
joint['audio_path'] = [_audio_path(c, f) for c, f in zip(joint['candidate_id'], joint['filename'])]

# ---------- Suspicion flags ----------
POS_THR = 0.30   # labeled cheating + both models say <0.30 -> probably honest
NEG_THR = 0.70   # labeled honest   + both models say >0.70 -> probably cheating
joint['suspect_type'] = ''
joint.loc[(joint['label_int']==1) & (joint['max_prob'] < POS_THR), 'suspect_type'] = 'pos_label_likely_honest'
joint.loc[(joint['label_int']==0) & (joint['min_prob'] > NEG_THR), 'suspect_type'] = 'neg_label_likely_cheating'

n_pos_total = int((joint['label_int']==1).sum())
n_neg_total = int((joint['label_int']==0).sum())
n_pos_susp  = int((joint['suspect_type']=='pos_label_likely_honest').sum())
n_neg_susp  = int((joint['suspect_type']=='neg_label_likely_cheating').sum())

BAR = '=' * 78
print(BAR)
print(' audios5 label-suspicion summary  (both models trained on audios2 only)')
print(BAR)
print(f'  total a5 audios scored : {len(joint)}')
pos_pct = n_pos_susp / max(n_pos_total, 1) * 100
neg_pct = n_neg_susp / max(n_neg_total, 1) * 100
print(f'  labeled cheating       : {n_pos_total}   suspect (both models <{POS_THR}): {n_pos_susp}  ({pos_pct:.1f}% of positives)')
print(f'  labeled honest         : {n_neg_total}   suspect (both models >{NEG_THR}): {n_neg_susp}  ({neg_pct:.1f}% of negatives)')

SHOW = 10
def _compact(df, sort_col, ascending, title):
    if not len(df):
        print()
        print(f'  (no rows for: {title})')
        return
    cols = ['candidate_id','filename','whisper','wavlm_ft','avg','audio_path']
    out = df.sort_values(sort_col, ascending=ascending).head(SHOW)[cols].copy()
    out['whisper']  = out['whisper'].round(3)
    out['wavlm_ft'] = out['wavlm_ft'].round(3)
    out['avg']      = out['avg'].round(3)
    print()
    print(f'--- TOP {SHOW}: {title} ---')
    with pd.option_context('display.max_columns', None, 'display.width', 240, 'display.max_colwidth', 80):
        print(out.to_string(index=False))

pos_df = joint[joint['label_int']==1].copy()
neg_df = joint[joint['label_int']==0].copy()
_compact(pos_df, 'min_prob', True,
         'a5 LABELED CHEATING but both models score lowest (label may be honest)')
_compact(neg_df, 'max_prob', False,
         'a5 LABELED HONEST but both models score highest (label may be cheating)')

# ---------- Per-candidate top-5 ----------
joint['disagree'] = (joint['label_int'] - joint['avg']).abs()
cand = joint.groupby('candidate_id').agg(
    n=('filename','size'),
    label_set=('label_int', lambda s: ','.join(map(str, sorted(s.unique())))),
    mean_disagree=('disagree','mean'),
    max_disagree =('disagree','max'),
).sort_values('mean_disagree', ascending=False).head(5)
print()
print('--- TOP 5 candidates by mean model-vs-label disagreement ---')
print(cand.round(3).to_string())

# ---------- Save full ranked CSVs ----------
out_path_pos = SAVE_DIR / 'a5_label_audit_positives.csv'
out_path_neg = SAVE_DIR / 'a5_label_audit_negatives.csv'
out_path_all = SAVE_DIR / 'a5_label_audit_all.csv'

cols_save = ['candidate_id','filename','label_int','whisper','wavlm_ft','avg',
             'min_prob','max_prob','suspect_type','audio_path']
pos_df.sort_values('min_prob').to_csv(out_path_pos, columns=cols_save, index=False)
neg_df.sort_values('max_prob', ascending=False).to_csv(out_path_neg, columns=cols_save, index=False)
joint.sort_values('disagree', ascending=False).to_csv(out_path_all, columns=cols_save, index=False)

print()
print('Saved:')
print(f'  {out_path_pos}    ({len(pos_df)} rows, sort by min_prob ASC -> most suspect on top)')
print(f'  {out_path_neg}    ({len(neg_df)} rows, sort by max_prob DESC -> most suspect on top)')
print(f'  {out_path_all}    ({len(joint)} rows, sorted by joint disagreement)')

print()
print('Decision rule:')
print(f'  - Re-listen first to the rows in {out_path_pos.name} where suspect_type is set.')
pct_int = int(round(pos_pct))
print(f'  - If {n_pos_susp} of {n_pos_total} positives ({pct_int}%) really are mislabeled,')
print('    fixing them and re-running section 6c should close most of the Rot B gap.')


## 12. Re-run rotations after audios5 relabel

Workflow:
1. Make sure §3 (the original rotations A and B) has been run at least once with the **original** `audios5GT.csv` — that produces the baseline this cell compares against.
2. Re-listen to the suspect rows in `checkpoints_honest_eval/a5_label_audit_negatives.csv` and edit `audios5GT.csv` directly: change the label column from `not cheating` to `cheating` (or whatever the project convention is) for any audio you confirmed is mislabeled.
3. Run this cell. It will:
   - back up the original `audios5GT.csv` (once, on first run) to `audios5GT_baseline.csv`,
   - snapshot the original rotation results to `rotation_A_baseline.csv` / `rotation_B_baseline.csv` if not already snapshotted,
   - reload audios5 with the corrected GT,
   - re-run rotations A and B,
   - print a before/after delta table per model.

The headline at the bottom is the one number that matters: whisper's Rot B F1 gap. If the relabel hypothesis is right, it should drop from ~+0.13 toward 0.

In [ ]:
import shutil

GT_PATH    = NB_DIR / 'audios5GT.csv'
GT_BACKUP  = NB_DIR / 'audios5GT_baseline.csv'
BASELINE_A = SAVE_DIR / 'rotation_A_baseline.csv'
BASELINE_B = SAVE_DIR / 'rotation_B_baseline.csv'
ROT_A      = SAVE_DIR / 'rotation_A.csv'
ROT_B      = SAVE_DIR / 'rotation_B.csv'

# --- 1. Snapshot the original GT once (so a revert is always possible) ---
if not GT_BACKUP.exists():
    if not GT_PATH.exists():
        raise RuntimeError(f'audios5GT.csv not found at {GT_PATH}')
    shutil.copy(GT_PATH, GT_BACKUP)
    print(f'  backed up original GT -> {GT_BACKUP.name}')
else:
    print(f'  GT backup already exists: {GT_BACKUP.name}  (untouched)')

# --- 2. Snapshot baseline rotation metrics once ---
if not BASELINE_A.exists():
    if not ROT_A.exists() or not ROT_B.exists():
        raise RuntimeError('Run section 3 first (writes rotation_A.csv and rotation_B.csv).')
    shutil.copy(ROT_A, BASELINE_A)
    shutil.copy(ROT_B, BASELINE_B)
    print(f'  snapshotted baseline rotations -> {BASELINE_A.name}, {BASELINE_B.name}')
else:
    print(f'  rotation baselines already snapshotted  (untouched)')

# --- 3. Reload audios5 from current (possibly corrected) GT ---
import re as _re_rerun
_RE_CAND_RR = _re_rerun.compile(r'^(.+)_(\d{1,3})\.[a-zA-Z0-9]+$')
print('Reloading audios5 from audios5GT.csv (current state)...')
batches['audios5'] = load_folder('audios5')
batches['audios5']['candidate_id'] = batches['audios5']['filename'].astype(str).map(
    lambda f: (_RE_CAND_RR.match(f).group(1) if _RE_CAND_RR.match(f) else None))
_y5 = batches['audios5']['label_int'].values
print(f'  audios5 now: n={len(_y5)}  cheat={int((_y5==1).sum())}  honest={int((_y5==0).sum())}')
_base_gt = pd.read_csv(GT_BACKUP)
_curr_gt = pd.read_csv(GT_PATH)
_n_changed = (_base_gt.set_index(_base_gt.columns[0]).iloc[:,0] !=
              _curr_gt.set_index(_curr_gt.columns[0]).iloc[:,0]).sum() if list(_base_gt.columns) == list(_curr_gt.columns) else 'unknown'
print(f'  rows changed vs baseline GT: {_n_changed}')

# --- 4. Re-run both rotations with current labels ---
print('Running Rotation A: train=[a2,a4] CV=a4 test=a5 ...')
df_A_corr, _ = run_rotation_base(['audios2','audios4'], 'audios4', 'audios5')
print('Running Rotation B: train=[a2,a5] CV=a5 test=a4 ...')
df_B_corr, _ = run_rotation_base(['audios2','audios5'], 'audios5', 'audios4')
df_A_corr.to_csv(SAVE_DIR / 'rotation_A_corrected.csv', index=False)
df_B_corr.to_csv(SAVE_DIR / 'rotation_B_corrected.csv', index=False)

# --- 5. Before/after compare on F1 strategy ---
def _compare(label, base_csv, corr_df):
    base = pd.read_csv(base_csv)
    base_f1 = base[base['strategy']=='F1'][['model','cv','te','gap']].rename(
        columns={'cv':'cv_base','te':'te_base','gap':'gap_base'})
    corr_f1 = corr_df[corr_df['strategy']=='F1'][['model','cv','te','gap']].rename(
        columns={'cv':'cv_corr','te':'te_corr','gap':'gap_corr'})
    m = base_f1.merge(corr_f1, on='model')
    m['cv_delta']  = (m['cv_corr']  - m['cv_base']).round(3)
    m['te_delta']  = (m['te_corr']  - m['te_base']).round(3)
    m['gap_delta'] = (m['gap_corr'] - m['gap_base']).round(3)
    print()
    print('='*90)
    print(f' {label}  (F1 strategy)')
    print('='*90)
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print(m.to_string(index=False))
    return m

A_cmp = _compare('ROTATION A baseline vs corrected (CV=a4 test=a5)', BASELINE_A, df_A_corr)
B_cmp = _compare('ROTATION B baseline vs corrected (CV=a5 test=a4)', BASELINE_B, df_B_corr)

# --- 6. Headline ---
print()
print('='*90)
print(' HEADLINE: did Rot B whisper gap shrink?  (the question this whole audit was for)')
print('='*90)
_w = B_cmp[B_cmp['model']=='whisper_wp_xgb'].iloc[0]
print(f'  whisper_wp_xgb Rot B F1 gap:')
print(f'    baseline   {_w["gap_base"]:+.3f}')
print(f'    corrected  {_w["gap_corr"]:+.3f}')
print(f'    delta      {_w["gap_delta"]:+.3f}   (negative = gap shrunk = relabel hypothesis confirmed)')
if 'wavlm_whole_ft' in B_cmp['model'].values:
    _v = B_cmp[B_cmp['model']=='wavlm_whole_ft'].iloc[0]
    print(f'  wavlm_whole_ft Rot B F1 gap:')
    print(f'    baseline   {_v["gap_base"]:+.3f}')
    print(f'    corrected  {_v["gap_corr"]:+.3f}')
    print(f'    delta      {_v["gap_delta"]:+.3f}')

print()
print('To revert the GT to the original:  shutil.copy(GT_BACKUP, GT_PATH)  then re-run this cell.')


## 13. Speaking-time per audio + per-batch distribution

The §11 audit surfaced a systematic confound: candidates whose audios are very short tend to either repeat the question or speak only a sentence, which acoustically looks scripted (low pause variance, fluent delivery) without the candidate actually cheating. Both whisper and wavlm_ft fire on these.

This cell computes per-audio:
- `total_duration_s` — file length, from `soundfile.info` (cheap, no audio loaded).
- `speaking_time_s` — sum of whisper segment durations if a `{batch}_transcripts.json` file is present; otherwise falls back to `n_words / words_per_sec` from `{batch}_features.csv` (already cached). Both are good approximations of "voiced time"; the transcript-based one is more accurate.
- `speech_ratio = speaking_time_s / total_duration_s`.

Caches to `checkpoints_honest_eval/{batch}_durations.csv` so the audio scan only happens once. Plots the distribution of `speaking_time_s` per batch, overlaid by class, with vertical lines at common thresholds (15 s / 25 s / 40 s).

In [ ]:
import soundfile as _sf
import json as _json

def _audio_dur(path):
    info = _sf.info(str(path))
    return info.frames / info.samplerate

def _load_transcripts(batch):
    p = NB_DIR / f'{batch}_transcripts.json'
    if not p.exists():
        return {}
    with open(p, 'r', encoding='utf-8') as f:
        t = _json.load(f)
    if isinstance(t, dict):
        return t
    if isinstance(t, list):
        out = {}
        for x in t:
            for k in ('filename', 'file', 'name'):
                if k in x:
                    out[x[k]] = x
                    break
        return out
    return {}

def _speech_from_segments(entry):
    segs = entry.get('segments') if isinstance(entry, dict) else None
    if not segs:
        return None
    try:
        return float(sum((s['end'] - s['start']) for s in segs))
    except (KeyError, TypeError):
        return None

def compute_durations(batch, force=False):
    cache = SAVE_DIR / f'{batch}_durations.csv'
    if cache.exists() and not force:
        return pd.read_csv(cache)
    df = batches[batch][['filename','candidate_id','label_int']].copy()
    transcripts = _load_transcripts(batch)
    feats = pd.read_csv(NB_DIR / f'{batch}_features.csv').set_index('filename')
    durs, speeches, sources = [], [], []
    for fn, cid in zip(df['filename'], df['candidate_id']):
        path = NB_DIR / batch / str(cid) / fn
        try:
            dur = _audio_dur(path)
        except Exception:
            dur = float('nan')
        durs.append(dur)
        # Speaking-time source preference: transcripts > derived from features
        sp, src = None, 'none'
        entry = transcripts.get(fn) or transcripts.get(str(fn))
        if entry is not None:
            sp = _speech_from_segments(entry)
            if sp is not None:
                src = 'transcript'
        if sp is None and fn in feats.index:
            n_w = feats.at[fn, 'n_words']    if 'n_words'       in feats.columns else None
            wps = feats.at[fn, 'words_per_sec'] if 'words_per_sec' in feats.columns else None
            if pd.notna(n_w) and pd.notna(wps) and wps > 0:
                sp, src = float(n_w) / float(wps), 'derived_words_per_sec'
        speeches.append(sp)
        sources.append(src)
    df['total_duration_s'] = durs
    df['speaking_time_s']  = speeches
    df['speech_ratio']     = df['speaking_time_s'] / df['total_duration_s']
    df['source']           = sources
    df.to_csv(cache, index=False)
    return df

print('Computing per-audio durations (cached on disk after first run)...')
durations = {b: compute_durations(b) for b in BATCHES}
for b, d in durations.items():
    src_counts = d['source'].value_counts().to_dict()
    miss_dur   = int(d['total_duration_s'].isna().sum())
    miss_sp    = int(d['speaking_time_s'].isna().sum())
    print(f'  {b}: n={len(d)}  source={src_counts}  missing_dur={miss_dur}  missing_speech={miss_sp}')

# Compact summary table
summary_rows = []
for b, d in durations.items():
    for cls, label in [(0, 'honest'), (1, 'cheat')]:
        sub = d[d['label_int']==cls]['speaking_time_s'].dropna()
        summary_rows.append({
            'batch':   b,
            'class':   label,
            'n':       len(sub),
            'min':     round(sub.min(), 1)  if len(sub) else None,
            'p10':     round(sub.quantile(0.10), 1) if len(sub) else None,
            'p25':     round(sub.quantile(0.25), 1) if len(sub) else None,
            'median':  round(sub.median(), 1)      if len(sub) else None,
            'p75':     round(sub.quantile(0.75), 1) if len(sub) else None,
            'max':     round(sub.max(), 1)         if len(sub) else None,
            'pct_under_25s': round((sub < 25).mean() * 100, 1) if len(sub) else None,
            'pct_under_15s': round((sub < 15).mean() * 100, 1) if len(sub) else None,
        })
summary_df = pd.DataFrame(summary_rows)
print()
print('Speaking-time distribution per batch x class (seconds):')
with pd.option_context('display.max_columns', None, 'display.width', 160):
    print(summary_df.to_string(index=False))

# Per-batch histograms
if HAS_MPL:
    fig, axes = plt.subplots(1, len(BATCHES), figsize=(5*len(BATCHES), 4), sharey=True)
    if len(BATCHES) == 1:
        axes = [axes]
    bins = np.arange(0, 121, 5)
    for ax, b in zip(axes, BATCHES):
        d = durations[b]
        h = d[d['label_int']==0]['speaking_time_s'].dropna()
        c = d[d['label_int']==1]['speaking_time_s'].dropna()
        ax.hist(h, bins=bins, alpha=0.55, label=f'honest n={len(h)}', color='C0')
        ax.hist(c, bins=bins, alpha=0.55, label=f'cheat n={len(c)}',  color='C3')
        for thr, style in [(15, ':'), (25, '--'), (40, '-.')]:
            ax.axvline(thr, color='k', linestyle=style, linewidth=0.8, alpha=0.6)
        ax.set_title(b)
        ax.set_xlabel('speaking_time_s')
        ax.legend(fontsize=8, loc='upper right')
    axes[0].set_ylabel('# audios')
    fig.suptitle('Speaking-time distribution by batch and class  (dotted=15s, dashed=25s, dashdot=40s)')
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available; skipping histograms')


## 14. Combined experiment — relabel + min-speaking-time filter, dual test views

This is the high-quality calibration cell (slow, multi-seed, all 5 strategies). Run it once with the chosen `MIN_SPEAKING_S` to get the precise post-intervention numbers.

What it does:
1. **Reloads `audios5` from disk** — picks up any `audios5GT.csv` edits you made after the §11 audit.
2. **Filters training-eligible batches** (always-train + CV target) at `MIN_SPEAKING_S`. Test batch left intact for full-test scoring.
3. **Runs both rotations** with the existing `run_rotation_base` (3 seeds × 5 folds × bootstrap CI per strategy).
4. **Refits each model on filtered training** to score the test set, then computes metrics two ways:
   - `te_full` — metrics on the **entire** test batch (no filter applied to test). This is the deployment-realistic number.
   - `te_filt` — metrics on the **subset** of test rows with `speaking_time_s ≥ MIN_SPEAKING_S`. This isolates "how good is the model on audios where the filter assumption holds?"
5. **Compares to baseline** = pre-relabel + pre-filter snapshot (`rotation_A.csv` / `rotation_B.csv` from §3's first run).

The delta between baseline and current = combined effect of (relabel + filter).
The delta between `te_full` and `te_filt` = how much the filter would help if it were also applied at deployment time (it isn't — but useful as a ceiling estimate).

Knobs at the top:
- `MIN_SPEAKING_S` — your chosen threshold (default 30, per §15 sweep).
- `MODELS_TO_RUN` — None for all 7 models (slow ~2 hr); pass a list to trim.
- `RELOAD_AUDIOS5_GT` — True to reload audios5 from disk (set False if you already reloaded earlier in the session).

In [ ]:
# ---------- KNOBS ----------
MIN_SPEAKING_S    = 30.0
MODELS_TO_RUN     = None     # None = all in BASE_REGISTRY; or list e.g. ['whisper_wp_xgb','wavlm_whole_ft','text_stylo']
RELOAD_AUDIOS5_GT = True     # reload audios5 from disk so any GT edits take effect

if 'durations' not in dir():
    raise RuntimeError('Run section 13 first (defines `durations`).')
if 'run_rotation_base' not in dir():
    raise RuntimeError('Run section 3 first (defines `run_rotation_base`).')

import re as _re14
_RE_CAND14 = _re14.compile(r'^(.+)_(\d{1,3})\.[a-zA-Z0-9]+$')

# ---------- 1. Reload audios5 to pick up GT corrections ----------
if RELOAD_AUDIOS5_GT:
    print('Reloading audios5 from disk to pick up any audios5GT.csv edits...')
    new_a5 = load_folder('audios5')
    new_a5['candidate_id'] = new_a5['filename'].astype(str).map(
        lambda f: (_RE_CAND14.match(f).group(1) if _RE_CAND14.match(f) else None))
    if 'label_int' in batches['audios5'].columns:
        merged = batches['audios5'][['filename','label_int']].merge(
            new_a5[['filename','label_int']], on='filename', suffixes=('_old','_new'))
        n_changed = int((merged['label_int_old'] != merged['label_int_new']).sum())
        print(f'  audios5 label changes vs in-memory copy: {n_changed} rows')
    batches['audios5'] = new_a5
    durations['audios5'] = compute_durations('audios5', force=True)
    print(f'  audios5 now: n={len(new_a5)}  cheat={int((new_a5["label_int"]==1).sum())}  honest={int((new_a5["label_int"]==0).sum())}')

# ---------- 2. Filter helper (NaN durations are kept) ----------
def _filter_batch(name, min_s):
    if min_s <= 0:
        return batches[name]
    d = durations[name][['filename','speaking_time_s']]
    keep = set(d[(d['speaking_time_s'] >= min_s) | (d['speaking_time_s'].isna())]['filename'])
    return batches[name][batches[name]['filename'].isin(keep)].reset_index(drop=True)

# ---------- 3. Filter stats ----------
filt_stats = []
for b in BATCHES:
    n_b = len(batches[b])
    n_a = len(_filter_batch(b, MIN_SPEAKING_S))
    pos_b = int((batches[b]['label_int']==1).sum())
    pos_a = int((_filter_batch(b, MIN_SPEAKING_S)['label_int']==1).sum())
    filt_stats.append({
        'batch':       b,
        'n_before':    n_b,
        'n_after':     n_a,
        'dropped':     n_b - n_a,
        'pct_dropped': round((n_b - n_a) / max(n_b,1) * 100, 1),
        'pos_before':  pos_b,
        'pos_after':   pos_a,
        'pos_dropped': pos_b - pos_a,
        'neg_dropped': (n_b - pos_b) - (n_a - pos_a),
    })
print()
print(f'Min speaking-time filter applied: keep audios with speaking_time_s >= {MIN_SPEAKING_S}')
print(pd.DataFrame(filt_stats).to_string(index=False))

# ---------- 4. Combined rotation: filtered training, dual-test scoring ----------
model_list = MODELS_TO_RUN or list(BASE_REGISTRY)
print()
print(f'Models in this run ({len(model_list)}): {model_list}')

def _dual_test_rotation(train_folders, cv_target, test_folder, min_s, models):
    """Filter training-eligible batches; score full + filtered test for each model (F1 strategy)."""
    view = dict(batches)
    for b in train_folders:
        view[b] = _filter_batch(b, min_s)
    df_cv  = view[cv_target].reset_index(drop=True)
    y_cv   = df_cv['label_int'].values
    groups = df_cv['candidate_id'].values
    df_al  = pd.concat([view[b] for b in train_folders if b != cv_target], ignore_index=True) \
             if any(b != cv_target for b in train_folders) else None
    df_te  = view[test_folder].reset_index(drop=True)   # unfiltered (full test)
    y_te   = df_te['label_int'].values
    sp_te  = durations[test_folder].set_index('filename')['speaking_time_s']
    mask_keep = df_te['filename'].map(
        lambda fn: pd.isna(sp_te.get(fn, np.nan)) or sp_te.get(fn, 0) >= min_s
    ).values
    rows = []
    for name in models:
        X_fn, factory = BASE_REGISTRY[name]
        oof = repeated_kfold_oof(df_cv, df_al, X_fn, factory)
        df_tr_full = pd.concat([df_al, df_cv], ignore_index=True) if df_al is not None else df_cv
        Xtr = X_fn(df_tr_full); ytr = df_tr_full['label_int'].values
        sc = StandardScaler().fit(Xtr)
        clf = factory(42); clf.fit(sc.transform(Xtr), ytr)
        p_te = clf.predict_proba(sc.transform(X_fn(df_te)))[:,1]
        thr_pt, cv_f1 = best_f1_thr(oof, y_cv)
        thr_md, thr_lo, thr_hi, _ = bootstrap_thr_ci(oof, y_cv, 'F1')
        thr_use = thr_md if thr_md is not None else thr_pt
        m_full = metrics_at(p_te, y_te, thr_use)
        if mask_keep.sum() > 0:
            m_filt = metrics_at(p_te[mask_keep], y_te[mask_keep], thr_use)
        else:
            m_filt = dict(prec=None, rec=None, f1=None)
        gap_full = (cv_f1 - m_full['f1']) if m_full['f1'] is not None else None
        gap_filt = (cv_f1 - m_filt['f1']) if m_filt['f1'] is not None else None
        rows.append({
            'model':         name,
            'thr':           round(thr_use, 3) if thr_use is not None else None,
            'thr_p10':       round(thr_lo, 3)  if thr_lo  is not None else None,
            'thr_p90':       round(thr_hi, 3)  if thr_hi  is not None else None,
            'cv':            round(cv_f1, 3),
            'te_full':       round(m_full['f1'], 3) if m_full['f1'] is not None else None,
            'te_prec_full':  round(m_full['prec'], 3) if m_full['prec'] is not None else None,
            'te_rec_full':   round(m_full['rec'], 3)  if m_full['rec']  is not None else None,
            'gap_full':      round(gap_full, 3) if gap_full is not None else None,
            'n_te_full':     int(len(y_te)),
            'te_filt':       round(m_filt['f1'], 3) if m_filt['f1'] is not None else None,
            'te_prec_filt':  round(m_filt['prec'], 3) if m_filt['prec'] is not None else None,
            'te_rec_filt':   round(m_filt['rec'], 3)  if m_filt['rec']  is not None else None,
            'gap_filt':      round(gap_filt, 3) if gap_filt is not None else None,
            'n_te_filt':     int(mask_keep.sum()),
        })
    return pd.DataFrame(rows)

import time as _t14
print()
print('Running Rotation A (train=[a2,a4] filtered, CV=a4, test=a5) ...')
_t0 = _t14.time()
df_A_dual = _dual_test_rotation(['audios2','audios4'], 'audios4', 'audios5', MIN_SPEAKING_S, model_list)
print(f'  done in {_t14.time()-_t0:.1f}s')

print('Running Rotation B (train=[a2,a5] filtered, CV=a5, test=a4) ...')
_t0 = _t14.time()
df_B_dual = _dual_test_rotation(['audios2','audios5'], 'audios5', 'audios4', MIN_SPEAKING_S, model_list)
print(f'  done in {_t14.time()-_t0:.1f}s')

df_A_dual.to_csv(SAVE_DIR / f'rotation_A_minS{int(MIN_SPEAKING_S)}_dual.csv', index=False)
df_B_dual.to_csv(SAVE_DIR / f'rotation_B_minS{int(MIN_SPEAKING_S)}_dual.csv', index=False)

# ---------- 5. Compare to baseline (pre-relabel, pre-filter) ----------
BASE_A_PATH = SAVE_DIR / 'rotation_A_baseline.csv' if (SAVE_DIR / 'rotation_A_baseline.csv').exists() else SAVE_DIR / 'rotation_A.csv'
BASE_B_PATH = SAVE_DIR / 'rotation_B_baseline.csv' if (SAVE_DIR / 'rotation_B_baseline.csv').exists() else SAVE_DIR / 'rotation_B.csv'

def _compare(label, base_path, dual_df, te_col, gap_col):
    base = pd.read_csv(base_path)
    base_f1 = base[base['strategy']=='F1'][['model','cv','te','gap']].rename(
        columns={'cv':'cv_base','te':'te_base','gap':'gap_base'})
    cur = dual_df[['model','cv', te_col, gap_col]].rename(
        columns={'cv':'cv_cur', te_col:'te_cur', gap_col:'gap_cur'})
    m = base_f1.merge(cur, on='model')
    m['cv_delta']  = (m['cv_cur']  - m['cv_base']).round(3)
    m['te_delta']  = (m['te_cur']  - m['te_base']).round(3)
    m['gap_delta'] = (m['gap_cur'] - m['gap_base']).round(3)
    print()
    print('=' * 110)
    print(f' {label}  (F1 strategy)')
    print('  baseline = pre-relabel, pre-filter (from rotation_*.csv)')
    print('=' * 110)
    with pd.option_context('display.max_columns', None, 'display.width', 220):
        print(m.to_string(index=False))
    return m

A_full = _compare(f'Rotation A vs baseline  -- TEST = full a5 (unfiltered, deployment-like)',
                  BASE_A_PATH, df_A_dual, 'te_full', 'gap_full')
A_filt = _compare(f'Rotation A vs baseline  -- TEST = a5 filtered to >= {MIN_SPEAKING_S}s only',
                  BASE_A_PATH, df_A_dual, 'te_filt', 'gap_filt')
B_full = _compare(f'Rotation B vs baseline  -- TEST = full a4 (unfiltered, deployment-like)',
                  BASE_B_PATH, df_B_dual, 'te_full', 'gap_full')
B_filt = _compare(f'Rotation B vs baseline  -- TEST = a4 filtered to >= {MIN_SPEAKING_S}s only',
                  BASE_B_PATH, df_B_dual, 'te_filt', 'gap_filt')

# ---------- 6. Headline ----------
print()
print('=' * 110)
print(f' HEADLINE  (combined effect: any GT relabel + filter at MIN_SPEAKING_S={MIN_SPEAKING_S}s)')
print('=' * 110)
for model_name in ['whisper_wp_xgb','wavlm_whole_ft','text_stylo']:
    if model_name not in df_B_dual['model'].values: continue
    rB_full = B_full[B_full['model']==model_name].iloc[0]
    rB_filt = B_filt[B_filt['model']==model_name].iloc[0]
    print(f'  {model_name}')
    print(f'    Rot B  full-test    gap  baseline {rB_full["gap_base"]:+.3f}  ->  current {rB_full["gap_cur"]:+.3f}   delta {rB_full["gap_delta"]:+.3f}')
    print(f'    Rot B  full-test    F1   baseline  {rB_full["te_base"]:.3f}  ->  current  {rB_full["te_cur"]:.3f}   delta {rB_full["te_delta"]:+.3f}')
    print(f'    Rot B  filtered-te  gap  baseline {rB_full["gap_base"]:+.3f}  ->  current {rB_filt["gap_cur"]:+.3f}   delta {rB_filt["gap_delta"]:+.3f}   (n={int(df_B_dual[df_B_dual["model"]==model_name].iloc[0]["n_te_filt"])} of {int(df_B_dual[df_B_dual["model"]==model_name].iloc[0]["n_te_full"])})')
    print(f'    Rot B  filtered-te  F1   baseline  {rB_full["te_base"]:.3f}  ->  current  {rB_filt["te_cur"]:.3f}   delta {rB_filt["te_delta"]:+.3f}')
    print()

print('Reading guide:')
print('  - full-test gap delta strongly negative -> intervention closed the deployment-realistic gap.')
print('  - filtered-te F1 >> full-test F1 -> the model is genuinely better on audios where it has enough signal.')
print('    The filtered-te number is what you would see if you abstained on short audios at deployment.')
print('  - cv_delta strongly negative without te_delta improving -> filter cut into real signal; threshold should drop.')


## 15. Fast min-speaking-time sweep

§14 takes ~2 hours per threshold because it runs all 7 base models × 3 seeds × 5 folds × bootstrap CIs × 2 rotations. This sweep is a slimmer version: a single F1 estimate per (model, rotation, threshold), no bootstrap, configurable folds/seeds/models. With the defaults it sweeps 6 thresholds in roughly 15-20 min on CPU.

The point of this cell is to find the *shape* of the gap-vs-threshold curve, not to produce final calibrated numbers. Once the optimal threshold is identified you go back to §14 with that one value for the precise estimate.

Speed knobs (top of the code cell — edit and re-run):
- `SWEEP_MODELS`: which models to include. Default skips wavlm_whole_ft (slow, follows whisper closely). Add it back if you want to confirm.
- `SWEEP_THRESHOLDS`: list of `MIN_SPEAKING_S` values to sweep. `0` = no filter (baseline).
- `SWEEP_N_FOLDS`: 3 for speed, 5 for parity with §3.
- `SWEEP_N_SEEDS`: 1 for speed, 3 for parity.
- `SWEEP_N_EST`: XGB n_estimators (200 fast / 400 default).

Test batch is always unfiltered. Same logic as §14: filter only training-eligible batches.

In [ ]:
# ----- Speed knobs (edit before running) -----
SWEEP_MODELS     = ['whisper_wp_xgb', 'text_stylo']        # add 'wavlm_whole_ft' for ~3x slower
SWEEP_THRESHOLDS = [0, 15, 20, 25, 30, 35]                  # 0 = no filter (baseline)
SWEEP_N_FOLDS    = 3                                        # 5 for parity with §3
SWEEP_N_SEEDS    = 1                                        # 3 for parity
SWEEP_N_EST      = 200                                      # 400 for parity

if 'durations' not in dir():
    raise RuntimeError('Run section 13 first (defines `durations`).')
if 'best_f1_thr' not in dir() or 'metrics_at' not in dir():
    raise RuntimeError('Run section 2 first (defines best_f1_thr / metrics_at).')

import time as _t

# Self-contained filter — independent of §14
def _sw_filter(name, min_s):
    if min_s <= 0:
        return batches[name]
    d = durations[name][['filename','speaking_time_s']]
    keep = set(d[(d['speaking_time_s'] >= min_s) | (d['speaking_time_s'].isna())]['filename'])
    return batches[name][batches[name]['filename'].isin(keep)].reset_index(drop=True)

# Fast XGB factory honoring the speed knobs
def _sw_xgb(n_feats, seed):
    cs = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=SWEEP_N_EST, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=cs, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=seed)

# Single-rotation runner — F1 strategy only, no bootstrap
def _sw_rot(view, train_folders, cv_target, test_folder, model_names):
    df_cv  = view[cv_target].reset_index(drop=True)
    y_cv   = df_cv['label_int'].values
    groups = df_cv['candidate_id'].values
    df_al  = pd.concat([view[b] for b in train_folders if b != cv_target], ignore_index=True) \
             if any(b != cv_target for b in train_folders) else None
    df_te  = view[test_folder].reset_index(drop=True)
    y_te   = df_te['label_int'].values
    rows = []
    for name in model_names:
        X_fn, _ = BASE_REGISTRY[name]
        n_feat  = X_fn(df_cv.head(1)).shape[1]
        oof_sum = np.zeros(len(df_cv)); oof_cnt = np.zeros(len(df_cv))
        for s in range(SWEEP_N_SEEDS):
            skf = StratifiedGroupKFold(n_splits=SWEEP_N_FOLDS, shuffle=True, random_state=42+s)
            for tr_idx, va_idx in skf.split(df_cv, y_cv, groups=groups):
                df_tr = pd.concat([df_al, df_cv.iloc[tr_idx]], ignore_index=True) \
                        if df_al is not None else df_cv.iloc[tr_idx]
                Xtr = X_fn(df_tr); ytr = df_tr['label_int'].values
                Xva = X_fn(df_cv.iloc[va_idx])
                sc = StandardScaler().fit(Xtr)
                clf = _sw_xgb(n_feat, 42+s)
                clf.fit(sc.transform(Xtr), ytr)
                oof_sum[va_idx] += clf.predict_proba(sc.transform(Xva))[:,1]
                oof_cnt[va_idx] += 1
        oof = oof_sum / np.maximum(oof_cnt, 1)
        df_tr_full = pd.concat([df_al, df_cv], ignore_index=True) if df_al is not None else df_cv
        Xtr = X_fn(df_tr_full); ytr = df_tr_full['label_int'].values
        sc = StandardScaler().fit(Xtr)
        clf = _sw_xgb(n_feat, 42); clf.fit(sc.transform(Xtr), ytr)
        p_te = clf.predict_proba(sc.transform(X_fn(df_te)))[:,1]
        thr, cv_f1 = best_f1_thr(oof, y_cv)
        te = metrics_at(p_te, y_te, thr)
        rows.append({
            'model': name, 'thr': round(thr,2),
            'cv_f1': round(cv_f1,3), 'te_f1': round(te['f1'],3),
            'gap':   round(cv_f1 - te['f1'], 3),
        })
    return pd.DataFrame(rows)

# ---- Sweep ----
results = []
for min_s in SWEEP_THRESHOLDS:
    t0 = _t.time()
    print(f'--- MIN_SPEAKING_S = {min_s}s ---')
    view = dict(batches)
    view['audios2'] = _sw_filter('audios2', min_s)
    view['audios4'] = _sw_filter('audios4', min_s)
    print(f'  Rot A  a2:{len(view["audios2"])}/{len(batches["audios2"])}  a4:{len(view["audios4"])}/{len(batches["audios4"])}  test_a5:{len(view["audios5"])}')
    df_A = _sw_rot(view, ['audios2','audios4'], 'audios4', 'audios5', SWEEP_MODELS)
    view = dict(batches)
    view['audios2'] = _sw_filter('audios2', min_s)
    view['audios5'] = _sw_filter('audios5', min_s)
    print(f'  Rot B  a2:{len(view["audios2"])}/{len(batches["audios2"])}  a5:{len(view["audios5"])}/{len(batches["audios5"])}  test_a4:{len(view["audios4"])}')
    df_B = _sw_rot(view, ['audios2','audios5'], 'audios5', 'audios4', SWEEP_MODELS)
    for r in df_A.itertuples():
        results.append({'min_s': min_s, 'rot': 'A', 'model': r.model,
                        'cv_f1': r.cv_f1, 'te_f1': r.te_f1, 'gap': r.gap})
    for r in df_B.itertuples():
        results.append({'min_s': min_s, 'rot': 'B', 'model': r.model,
                        'cv_f1': r.cv_f1, 'te_f1': r.te_f1, 'gap': r.gap})
    print(f'  done in {_t.time()-t0:.1f}s')

results_df = pd.DataFrame(results)
results_df.to_csv(SAVE_DIR / 'min_speaking_sweep.csv', index=False)

def _piv(metric):
    return results_df.pivot_table(index=['model','rot'], columns='min_s', values=metric).round(3)

print()
print('='*90)
print(' GAP (cv - te)  per (model, rotation) across MIN_SPEAKING_S')
print('='*90)
print(_piv('gap').to_string())
print()
print('='*90)
print(' TEST F1  per (model, rotation) across MIN_SPEAKING_S')
print('='*90)
print(_piv('te_f1').to_string())
print()
print('='*90)
print(' CV F1  per (model, rotation) across MIN_SPEAKING_S')
print('='*90)
print(_piv('cv_f1').to_string())

print()
print(f'Saved -> {SAVE_DIR / "min_speaking_sweep.csv"}')
print()
print('Reading guide:')
print('  - Find the column where Rot B gap is smallest AND test F1 is highest.')
print('  - Monotonic gap shrink through 35 -> stricter is better; we are still removing confounds.')
print('  - U-shape -> optimum somewhere in middle; past it we cut into real signal.')
print('  - Sharp cv_f1 drop at any threshold -> training set too small at that level.')
